# Metaheuristic Enhanced Model for Financial Markets

This notebook combines **Feature Selection** and **Feature Engineering** using Genetic Algorithms to enhance machine learning models for financial market trend prediction. 

## Workflow Overview:

0. **Settings**
1. **Installs and Imports**
2. **Data Cleaning**
3. **Baseline Feature Engineering** 
4. **Data Preprocessing** 
5. **Baseline Model Training and Testing** 
6. **Feature Selection using Genetic Algorithm (GA)**
7. **Feature Engineering using Genetic Algorithm (GA)**
8. **Enhanced Model Training and Testing**
9. **Comparison of Baseline and Enhanced Models**
10. **Save Results and Metadata**


## 0. Settings
set the TICKER to a valid ticker searchable by the polygon API. Should you wish to use a specific dataset collected from a different source, set the TICKER to none and the DATASET to the filename under raw_data

In [ ]:
TICKER = 'C:AUDZAR'  # Change this to your desired ticker (e.g., 'C:GBPUSD', 'C:USDJPY', etc.)
DATASET = None

## 1. Installs and Imports

First we install the requirements for the notebook and import the libraries

In [1]:
# Install dependencies from requirements.txt
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Import required libraries
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
import time
import pickle
import json

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Statistical and mathematical tools
from scipy import stats

# Machine learning
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Neural Networks
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

# Import our custom modules
from MH_Feature_Selection import GeneticAlgorithm
from MH_Feature_Engineering import engineer_adaptive_moving_average, engineer_fractal_dimension_indicator

# Suppress warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_style('whitegrid')

# For reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("All libraries imported successfully!")
print("Custom metaheuristics modules loaded!")

c:\Users\nicol\anaconda3\envs\research_proj_2\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\nicol\anaconda3\envs\research_proj_2\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


All libraries imported successfully!
Custom metaheuristics modules loaded!


### 1.1 Store Notebook Data

All the outputs and settings of various models in the notebook are stored in a timestamped folder for statistical comparisons later.

In [3]:
# Create timestamped directory for this run
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_directory = f"run_{timestamp}"
os.makedirs(run_directory, exist_ok=True)

print(f"Created run directory: {run_directory}")

# Create a README file to document this run
readme_content = f"""# Combined Metaheuristics Workflow Run
Run Date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
Timestamp: {timestamp}

## Workflow Components:
1. Data Loading and Cleaning
2. Baseline Feature Engineering  
3. Data Preprocessing (Time-series splits)
4. Baseline Model Training (RF, XGBoost, ANN)
5. GA Feature Selection
6. GA Feature Engineering (AMA, FDI)
7. Enhanced Model Training
8. Performance Comparison
9. Results Saving

## Files in this directory:
- *_baseline.pkl: Baseline models
- *_enhanced.pkl: Enhanced models with GA features
- selected_features_*.pkl: GA-selected feature subsets
- engineered_features_*.pkl: GA-engineered features
- X_*_scaled.pkl: Preprocessed data splits
- y_*.pkl: Target variable splits
- feature_names.pkl: Original feature names
- performance_comparison.csv: Results comparison
- README.txt: This file
"""

with open(os.path.join(run_directory, 'README.txt'), 'w') as f:
    f.write(readme_content)

print("Run documentation created!")

Created run directory: run_20251005_133636
Run documentation created!


### 1.2 Fetch Data
If a specific dataset is provided then that dataset will be used
If a specific ticker is provided, then we will first check if any data exists locally for that ticker. 
    If not, then data is downloaded.
    If yes, then that dataset is first updated to ensure the latest hours data in in the dataset.

In [ ]:
# if DATASET is not None, load it and skip fetching
if DATASET is not None:
    print(f"Loading dataset from provided DATASET variable...")
    data = pd.read_csv(DATASET)
    print(f"Dataset loaded with shape: {data.shape}")
else:

    # Ticker configuration - MODIFY THIS TO CHANGE TICKER
    TICKER_FILENAME = TICKER.replace(':', '_') + '_H1.csv'  # Convert to filename format

    # Polygon.io API configuration
    API_KEY = 'ySVTqEYU5OoF_j9Ic5mufeVhvvMFyhLt'  # Your Polygon.io API key
    MINIMUM_MONTHS = 6  # Minimum months of data to fetch
    BUFFER_HOURS = 100  # Additional hours for lookback periods

    # Timezone configuration - UPDATE THIS FOR YOUR LOCAL TIMEZONE
    LOCAL_TIMEZONE_OFFSET = 2  # Hours ahead of UTC (UTC+2 for your location)

    print(f"Processing ticker: {TICKER}")
    print(f"Expected filename: {TICKER_FILENAME}")
    print(f"Local timezone: UTC+{LOCAL_TIMEZONE_OFFSET:02d}:00")

    # Ensure raw_data directory exists
    os.makedirs('raw_data', exist_ok=True)

    # Check if ticker data exists
    ticker_path = os.path.join('raw_data', TICKER_FILENAME)
    data_exists = os.path.exists(ticker_path)

    if data_exists:
        print(f"Found existing data file: {TICKER_FILENAME}")
        
        # Load existing data
        data = pd.read_csv(ticker_path)
        print(f"Loaded data shape: {data.shape}")
        
        # Convert datetime column (assuming first column is datetime)
        datetime_col = data.columns[0]
        data[datetime_col] = pd.to_datetime(data[datetime_col])
        
        # Check latest timestamp
        latest_timestamp = data[datetime_col].max()
        
        # Get current time in both UTC and local timezone for comparison
        current_time_utc = pd.Timestamp.now(tz='UTC').tz_localize(None)
        current_time_local = current_time_utc + pd.Timedelta(hours=LOCAL_TIMEZONE_OFFSET)
        
        print(f"Latest data timestamp (UTC): {latest_timestamp}")
        print(f"Current time (UTC): {current_time_utc}")
        print(f"Current time (Local UTC+{LOCAL_TIMEZONE_OFFSET:02d}): {current_time_local}")
        
        # Calculate hours behind using UTC times (both data and API are in UTC)
        # But show local time context for user understanding
        hours_behind = (current_time_utc - latest_timestamp).total_seconds() / 3600
        hours_behind_local_context = (current_time_local - latest_timestamp).total_seconds() / 3600
        
        print(f"Data is {hours_behind:.1f} hours behind current UTC time")
        print(f"Data is {hours_behind_local_context:.1f} hours behind your local time (UTC+{LOCAL_TIMEZONE_OFFSET:02d})")
        
        # Use UTC-based calculation for API update decisions since both data and API use UTC
        if hours_behind > 2:
            print("Data needs updating. Fetching latest data...")
            
            # Smart update fetching - maximize each API call
            # Create a global variable to store the last successful timestamp
            if 'last_successful_timestamp' not in globals():
                global last_successful_timestamp
                last_successful_timestamp = None

            def smart_fetch_update(ticker, start_dt, end_dt, api_key):
                """Intelligently fetch update data with minimal API calls"""
                import requests
                import time
                global last_successful_timestamp
                
                # Use the last successful timestamp if available
                if last_successful_timestamp is not None:
                    print(f"📌 Resuming from last successful timestamp: {last_successful_timestamp}")
                    current_start = last_successful_timestamp + pd.Timedelta(hours=1)
                else:
                    current_start = start_dt
                
                all_data = []
                calls_made = 0
                start_time = time.time()
                max_calls_per_minute = 5
                
                print(f"📡 Smart update fetch from {current_start} to {end_dt} (UTC)")
                
                try:
                    while current_start < end_dt:
                        # Check if we need to wait for rate limit reset
                        elapsed = time.time() - start_time
                        if calls_made >= max_calls_per_minute and elapsed < 60:
                            wait_time = 60 - elapsed
                            print(f"⏳ Rate limit: waiting {wait_time:.0f}s before next batch...")
                            time.sleep(wait_time)
                            calls_made = 0
                            start_time = time.time()
                        
                        # Try to get as much data as possible in one call
                        from_date = current_start.strftime('%Y-%m-%d')
                        to_date = end_dt.strftime('%Y-%m-%d')
                        
                        print(f"🔄 API Call {calls_made + 1}: {from_date} to {to_date}")
                        
                        url = f'https://api.polygon.io/v2/aggs/ticker/{ticker}/range/1/hour/{from_date}/{to_date}?limit=1000&apiKey={api_key}'
                        
                        try:
                            response = requests.get(url)
                            api_data = response.json()
                            calls_made += 1
                            
                            if response.status_code == 200 and 'results' in api_data and api_data['results']:
                                records = api_data['results']
                                print(f"  ✅ Got {len(records)} records")
                                
                                # Check if we got less than the maximum, meaning we got all available data
                                if len(records) < 1000:
                                    print(f"  ℹ️ Got all available data ({len(records)} < 1000 limit)")
                                    all_data.extend(records)
                                    # Clear the last timestamp as we're done
                                    last_successful_timestamp = None
                                    break
                                else:
                                    # We hit the 1000 record limit, find the last timestamp and continue from there
                                    all_data.extend(records)
                                    last_timestamp = pd.to_datetime(records[-1]['t'], unit='ms')
                                    current_start = last_timestamp + pd.Timedelta(hours=1)
                                    # Save the last timestamp in case we need to resume
                                    last_successful_timestamp = last_timestamp
                                    print(f"  ⏭️ Hit 1000 record limit, continuing from {current_start}")
                                    print(f"  📊 Total records so far: {len(all_data)}")
                            
                            elif response.status_code == 429:  # Rate limit exceeded
                                print(f"  ⏳ Rate limit hit, waiting 60 seconds...")
                                time.sleep(60)
                                calls_made = 0  # Reset counter
                                start_time = time.time()
                                continue  # Retry this call
                            
                            else:
                                print(f"  ❌ API error: Status {response.status_code}")
                                if 'error' in api_data:
                                    print(f"     {api_data['error']}")
                                break
                            
                            # Wait between calls to respect rate limits
                            if current_start < end_dt:
                                print(f"  ⏱️ Waiting 15 seconds before next call...")
                                time.sleep(15)
                        
                        except Exception as e:
                            print(f"  ❌ Error: {e}")
                            # Keep the last timestamp so we can resume
                            break
                    
                except KeyboardInterrupt:
                    print("\n⚠️ Process interrupted by user. Progress saved.")
                    # The last_successful_timestamp will be preserved for next run
                
                print(f"📊 Update complete: {len(all_data)} total records from {calls_made} API calls")
                if last_successful_timestamp is not None:
                    print(f"💾 Progress saved. Resume point: {last_successful_timestamp}")
                return all_data
            
            # Fetch update data using UTC times (API expects UTC)
            update_results = smart_fetch_update(TICKER, latest_timestamp, current_time_utc, API_KEY)
            
            if update_results:
                print(f"Processing {len(update_results)} update records...")
                
                # Parse new data
                new_df = pd.DataFrame(update_results)
                new_df['timestamp'] = pd.to_datetime(new_df['t'], unit='ms')
                new_df = new_df[['timestamp', 'o', 'h', 'l', 'c', 'v']].rename(columns={
                    'timestamp': datetime_col,
                    'o': 'Open',
                    'h': 'High', 
                    'l': 'Low',
                    'c': 'Close',
                    'v': 'Volume'
                })
                
                # Remove duplicates and append new data
                new_df = new_df[new_df[datetime_col] > latest_timestamp]
                
                if len(new_df) > 0:
                    # Append new data
                    updated_data = pd.concat([data, new_df], ignore_index=True)
                    updated_data = updated_data.drop_duplicates(subset=[datetime_col]).sort_values(datetime_col)
                    
                    # Save updated data
                    updated_data.to_csv(ticker_path, index=False)
                    data = updated_data
                    print(f"✅ Updated data with {len(new_df)} new records")
                    print(f"📈 New data shape: {data.shape}")
                    
                    # Show updated timing info
                    new_latest = data[datetime_col].max()
                    new_hours_behind = (current_time_utc - new_latest).total_seconds() / 3600
                    new_hours_behind_local = (current_time_local - new_latest).total_seconds() / 3600
                    print(f"📊 Updated: Data now {new_hours_behind:.1f} hours behind UTC ({new_hours_behind_local:.1f} hours behind local time)")
                else:
                    print("ℹ️ No new data available after filtering")
            else:
                print("❌ No update data returned")
        else:
            print("Data is up to date")
            
    else:
        print(f"No existing data found for {TICKER}. Fetching historical data...")
        
        # Calculate date range for minimum data requirement using UTC
        current_time_utc = pd.Timestamp.now(tz='UTC').tz_localize(None)
        current_time_local = current_time_utc + pd.Timedelta(hours=LOCAL_TIMEZONE_OFFSET)
        start_date = current_time_utc - pd.DateOffset(months=MINIMUM_MONTHS) - pd.DateOffset(hours=BUFFER_HOURS)
        
        print(f"🎯 Target date range (UTC): {start_date.strftime('%Y-%m-%d')} to {current_time_utc.strftime('%Y-%m-%d')}")
        print(f"🕒 Your local time: {current_time_local.strftime('%Y-%m-%d %H:%M')} (UTC+{LOCAL_TIMEZONE_OFFSET:02d})")
        
        # Smart fetching function that maximizes each API call
        def smart_fetch_historical(ticker, start_dt, end_dt, api_key, max_calls_per_minute=5):
            """Intelligently fetch historical data with minimal API calls"""
            import requests
            import time
            
            all_data = []
            current_start = start_dt
            calls_made = 0
            start_time = time.time()
            
            print(f"🚀 Starting smart historical fetch...")
            print(f"📋 Strategy: Max {max_calls_per_minute} calls per minute, 50K records per call")
            
            while current_start < end_dt:
                # Check if we need to wait for rate limit reset
                elapsed = time.time() - start_time
                if calls_made >= max_calls_per_minute and elapsed < 60:
                    wait_time = 60 - elapsed
                    print(f"⏳ Rate limit: waiting {wait_time:.0f}s before next batch...")
                    time.sleep(wait_time)
                    calls_made = 0
                    start_time = time.time()
                
                # Try to get maximum data in single call
                from_date = current_start.strftime('%Y-%m-%d')
                to_date = end_dt.strftime('%Y-%m-%d')
                
                print(f"🔄 API Call {calls_made + 1}: {from_date} to {to_date}")
                
                url = f'https://api.polygon.io/v2/aggs/ticker/{ticker}/range/1/hour/{from_date}/{to_date}?limit=50000&apiKey={api_key}'
                
                try:
                    response = requests.get(url)
                    api_data = response.json()
                    calls_made += 1
                    
                    if response.status_code == 200:
                        if 'results' in api_data and api_data['results']:
                            records = api_data['results']
                            print(f"  ✅ Got {len(records)} records")
                            all_data.extend(records)
                            
                            # Check if we got all available data or hit the limit
                            if len(records) < 50000:
                                print(f"  🎉 Got all available data for requested period!")
                                break
                            else:
                                # We hit the 50K limit, continue from the last timestamp
                                last_timestamp = pd.to_datetime(records[-1]['t'], unit='ms')
                                current_start = last_timestamp + pd.Timedelta(hours=1)
                                print(f"  ⏭️ Hit 50K limit, continuing from {current_start.strftime('%Y-%m-%d %H:%M')}")
                        else:
                            print(f"  ℹ️ No data returned for period {from_date} to {to_date}")
                            break
                    
                    elif response.status_code == 429:  # Rate limit exceeded
                        print(f"  ⏳ Rate limit exceeded, waiting 60 seconds...")
                        time.sleep(60)
                        calls_made = 0  # Reset counter
                        start_time = time.time()
                        continue  # Retry this call
                    
                    else:
                        print(f"  ❌ API error: Status {response.status_code}")
                        if 'message' in api_data:
                            print(f"     Message: {api_data['message']}")
                        if 'error' in api_data:
                            print(f"     Error: {api_data['error']}")
                        
                        # For free tier limitations, break and use what we have
                        if response.status_code == 403:
                            print(f"  🔒 Free tier limit reached. Using {len(all_data)} records collected so far.")
                            break
                    
                    # Small delay between calls within the same minute
                    if current_start < end_dt:
                        time.sleep(2)  # 2-second delay between calls
                
                except Exception as e:
                    print(f"  ❌ Exception: {e}")
                    break
            
            print(f"📊 Historical fetch complete: {len(all_data)} total records from {calls_made} API calls")
            return all_data
        
        # Fetch historical data with smart strategy using UTC times
        all_results = smart_fetch_historical(TICKER, start_date, current_time_utc, API_KEY)
        
        if all_results:
            print(f"🔄 Processing {len(all_results)} historical records...")
            
            # Parse all data
            data = pd.DataFrame(all_results)
            data['timestamp'] = pd.to_datetime(data['t'], unit='ms')
            data = data[['timestamp', 'o', 'h', 'l', 'c', 'v']].rename(columns={
                'timestamp': 'Datetime',
                'o': 'Open',
                'h': 'High',
                'l': 'Low', 
                'c': 'Close',
                'v': 'Volume'
            })
            
            # Remove duplicates and sort by timestamp
            initial_count = len(data)
            data = data.drop_duplicates(subset=['Datetime']).sort_values('Datetime').reset_index(drop=True)
            final_count = len(data)
            
            if initial_count != final_count:
                print(f"🧹 Removed {initial_count - final_count} duplicate records")
            
            # Save data
            data.to_csv(ticker_path, index=False)
            print(f"💾 Saved {len(data)} unique records to {TICKER_FILENAME}")
            print(f"📈 Data shape: {data.shape}")
            print(f"📅 Date range (UTC): {data['Datetime'].min()} to {data['Datetime'].max()}")
            
            # Show local time context for the data range
            data_start_local = data['Datetime'].min() + pd.Timedelta(hours=LOCAL_TIMEZONE_OFFSET)
            data_end_local = data['Datetime'].max() + pd.Timedelta(hours=LOCAL_TIMEZONE_OFFSET)
            print(f"📅 Date range (Local UTC+{LOCAL_TIMEZONE_OFFSET:02d}): {data_start_local} to {data_end_local}")
            
            # Calculate coverage statistics
            actual_days = (data['Datetime'].max() - data['Datetime'].min()).days
            expected_hours = actual_days * 24
            fx_trading_hours = int(expected_hours * 0.7)  # FX markets trade ~70% of the time
            coverage = (len(data) / fx_trading_hours) * 100 if fx_trading_hours > 0 else 0
            
            print(f"📊 Coverage Analysis:")
            print(f"   • Actual period: {actual_days} days")
            print(f"   • Expected FX trading hours: ~{fx_trading_hours}")
            print(f"   • Actual records: {len(data)}")
            print(f"   • Coverage: {coverage:.1f}%")
            
            if len(data) < 1000:
                print(f"⚠️  Warning: Only {len(data)} records fetched. This might be due to free tier limitations.")
                print(f"   Consider upgrading to a paid plan for more complete historical data.")
            
        else:
            print("❌ Failed to fetch any historical data")
            raise Exception("Failed to fetch ticker data from Polygon.io")

print(f"Initial data shape: {data.shape}")
print(f"Columns: {len(data.columns)}")
print(f"Memory usage: {data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Display first few rows and basic info
print("\nFirst 5 rows:")
print(data.head())

print("\nData types:")
print(data.dtypes.value_counts())

Processing ticker: C:AUDZAR
Expected filename: C_AUDZAR_H1.csv
Local timezone: UTC+02:00
Found existing data file: C_AUDZAR_H1.csv
Loaded data shape: (2461, 6)
Latest data timestamp (UTC): 2025-08-22 16:00:00
Current time (UTC): 2025-10-05 11:57:24.139168
Current time (Local UTC+02): 2025-10-05 13:57:24.139168
Data is 1052.0 hours behind current UTC time
Data is 1054.0 hours behind your local time (UTC+02)
Data needs updating. Fetching latest data...
📡 Smart update fetch from 2025-08-22 16:00:00 to 2025-10-05 11:57:24.139168 (UTC)
🔄 API Call 1: 2025-08-22 to 2025-10-05
  ✅ Got 741 records
  ℹ️ Got all available data (741 < 50000 limit)
📊 Update complete: 741 total records from 1 API calls
Processing 741 update records...
✅ Updated data with 724 new records
📈 New data shape: (3185, 6)
📊 Updated: Data now 40.0 hours behind UTC (42.0 hours behind local time)
Initial data shape: (3185, 6)
Columns: 6
Memory usage: 0.15 MB

First 5 rows:
             Datetime       Open       High        Low

## 2. Data cleaning
- Handle missing values with fill forward and fill backward
- Handle infinite values by replacing with NaN and filling
- Remove any remaining Nan rows

In [ ]:
# Data cleaning and validation
print("Performing data cleaning...")

# Check for missing values
missing_values = data.isnull().sum()
print(f"\nMissing values per column:")
print(missing_values[missing_values > 0])

# Check for infinite values
numeric_cols = data.select_dtypes(include=[np.number]).columns
inf_values = np.isinf(data[numeric_cols]).sum()
print(f"\nInfinite values per column:")
print(inf_values[inf_values > 0])

# Handle missing values by forward fill and backward fill
if missing_values.sum() > 0:
    print(f"Filling {missing_values.sum()} missing values...")
    data = data.fillna(method='ffill').fillna(method='bfill')

# Handle infinite values by replacing with NaN and then filling
if inf_values.sum() > 0:
    print(f"Replacing {inf_values.sum()} infinite values...")
    data[numeric_cols] = data[numeric_cols].replace([np.inf, -np.inf], np.nan)
    data = data.fillna(method='ffill').fillna(method='bfill')

# Remove any remaining NaN rows
initial_rows = len(data)
data = data.dropna()
dropped_rows = initial_rows - len(data)
if dropped_rows > 0:
    print(f"Dropped {dropped_rows} rows with remaining NaN values")

print(f"\nCleaned data shape: {data.shape}")
print("Data cleaning completed!")

In [ ]:
# Rename columns to match required OHLCV format
data.columns = ['Datetime', 'Open', 'High', 'Low', 'Close', 'Volume']

# Parse datetime if present (adjust column name as needed)
if 'Date' in data.columns:
    data['Date'] = pd.to_datetime(data['Date'])
    data = data.set_index('Date')
elif 'Datetime' in data.columns:
    data['Datetime'] = pd.to_datetime(data['Datetime'])
    data = data.set_index('Datetime')
elif 'Timestamp' in data.columns:
    data['Timestamp'] = pd.to_datetime(data['Timestamp'])
    data = data.set_index('Timestamp')

print(f"Data date range: {data.index[0]} to {data.index[-1]}")

# Display basic statistics for OHLCV data
print("\nOHLCV basic statistics:")
print(data[['Open', 'High', 'Low', 'Close', 'Volume']].describe())

## 2.2 Date Range
Set a daterange to be used
In this case; 6 months of data + 100 hours to account for lookback windows

In [ ]:
# Define the date range or number of months for filtering
# Use the last N months of data plus 
months_to_use = 6
end_date_dynamic = data.index.max()
start_date_dynamic = end_date_dynamic - timedelta(days=months_to_use * 30)

# add a buffer of 100 hours to account for lookback periods in feature engineering
start_date_dynamic -= timedelta(hours=100)

# Filter the data based on the chosen option
#filtered_data = data.loc[start_date:end_date]  # Use this for specific date range
filtered_data = data.loc[start_date_dynamic:end_date_dynamic]  # Use this for last N months

print(f"Filtered data shape: {filtered_data.shape}")
print(f"Filtered data date range: {filtered_data.index.min()} to {filtered_data.index.max()}")

## 3. Baseline Feature Engineering

Apply comprehensive feature engineering to create technical indicators and the target variable from raw OHLCV data.

In [ ]:
# Create comprehensive technical indicators from raw OHLCV data
print("Creating comprehensive technical indicators...")

# Create a deep copy to work with for feature engineering
feature_df = filtered_data.copy()

# Define common lookback periods
lookback_periods = [5, 10, 20, 50, 100]

## 1. Basic Price Transformations

# Calculate returns (percentage change)
feature_df['Percentage_Change'] = feature_df['Close'].pct_change()
feature_df['Log_Return'] = np.log(feature_df['Close'] / feature_df['Close'].shift(1))

# Calculate price differences
feature_df['Price_Diff'] = feature_df['Close'].diff()
feature_df['Open_Close_Diff'] = feature_df['Close'] - feature_df['Open']
feature_df['High_Low_Diff'] = feature_df['High'] - feature_df['Low']

# Calculate price ratios
feature_df['High_Close_Ratio'] = feature_df['High'] / feature_df['Close']
feature_df['Low_Close_Ratio'] = feature_df['Low'] / feature_df['Close']
feature_df['Open_Close_Ratio'] = feature_df['Open'] / feature_df['Close']

# Calculate candle characteristics
feature_df['Candle_Range'] = feature_df['High'] - feature_df['Low']  # Total range
feature_df['Body_Size'] = abs(feature_df['Close'] - feature_df['Open'])  # Body size
feature_df['Upper_Shadow'] = feature_df['High'] - feature_df[['Open', 'Close']].max(axis=1)  # Upper shadow
feature_df['Lower_Shadow'] = feature_df[['Open', 'Close']].min(axis=1) - feature_df['Low']  # Lower shadow
feature_df['Body_To_Range_Ratio'] = feature_df['Body_Size'] / feature_df['Candle_Range']  # Body to range ratio

print("Basic price transformation features created")

## 2. Moving Averages and Derivatives

# Simple Moving Averages (SMA)
for period in lookback_periods:
    feature_df[f'SMA_{period}'] = feature_df['Close'].rolling(window=period).mean()
    feature_df[f'SMA_Dist_{period}'] = (feature_df['Close'] - feature_df[f'SMA_{period}']) / feature_df[f'SMA_{period}'] * 100

# Exponential Moving Averages (EMA)
for period in lookback_periods:
    feature_df[f'EMA_{period}'] = feature_df['Close'].ewm(span=period, adjust=False).mean()
    feature_df[f'EMA_Dist_{period}'] = (feature_df['Close'] - feature_df[f'EMA_{period}']) / feature_df[f'EMA_{period}'] * 100

# Moving Average Convergence Divergence (MACD)
feature_df['MACD_Line'] = feature_df['Close'].ewm(span=12, adjust=False).mean() - feature_df['Close'].ewm(span=26, adjust=False).mean()
feature_df['MACD_Signal'] = feature_df['MACD_Line'].ewm(span=9, adjust=False).mean()
feature_df['MACD_Histogram'] = feature_df['MACD_Line'] - feature_df['MACD_Signal']
feature_df['MACD_CrossAbove'] = ((feature_df['MACD_Line'] > feature_df['MACD_Signal']) & 
                                (feature_df['MACD_Line'].shift(1) <= feature_df['MACD_Signal'].shift(1))).astype(int)
feature_df['MACD_CrossBelow'] = ((feature_df['MACD_Line'] < feature_df['MACD_Signal']) & 
                                (feature_df['MACD_Line'].shift(1) >= feature_df['MACD_Signal'].shift(1))).astype(int)

# Moving Average Crossovers
feature_df['SMA_5_10_Cross'] = ((feature_df['SMA_5'] > feature_df['SMA_10']) & 
                               (feature_df['SMA_5'].shift(1) <= feature_df['SMA_10'].shift(1))).astype(int)
feature_df['SMA_10_20_Cross'] = ((feature_df['SMA_10'] > feature_df['SMA_20']) & 
                                (feature_df['SMA_10'].shift(1) <= feature_df['SMA_20'].shift(1))).astype(int)

# Triple Exponential Moving Average (TEMA)
for period in [10, 20, 50]:
    ema1 = feature_df['Close'].ewm(span=period, adjust=False).mean()
    ema2 = ema1.ewm(span=period, adjust=False).mean()
    ema3 = ema2.ewm(span=period, adjust=False).mean()
    feature_df[f'TEMA_{period}'] = 3 * ema1 - 3 * ema2 + ema3
    feature_df[f'TEMA_Dist_{period}'] = (feature_df['Close'] - feature_df[f'TEMA_{period}']) / feature_df[f'TEMA_{period}'] * 100

# Bollinger Bands
for period in [20]:
    feature_df[f'BB_Middle_{period}'] = feature_df['Close'].rolling(window=period).mean()
    feature_df[f'BB_Std_{period}'] = feature_df['Close'].rolling(window=period).std()
    feature_df[f'BB_Upper_{period}'] = feature_df[f'BB_Middle_{period}'] + 2 * feature_df[f'BB_Std_{period}']
    feature_df[f'BB_Lower_{period}'] = feature_df[f'BB_Middle_{period}'] - 2 * feature_df[f'BB_Std_{period}']
    feature_df[f'BB_Width_{period}'] = (feature_df[f'BB_Upper_{period}'] - feature_df[f'BB_Lower_{period}']) / feature_df[f'BB_Middle_{period}']
    feature_df[f'BB_Pct_B_{period}'] = (feature_df['Close'] - feature_df[f'BB_Lower_{period}']) / (feature_df[f'BB_Upper_{period}'] - feature_df[f'BB_Lower_{period}'])

print("Moving Average features created")

## 3. Volatility Indicators

# Average True Range (ATR)
for period in [14, 20]:
    high_low = feature_df['High'] - feature_df['Low']
    high_close = abs(feature_df['High'] - feature_df['Close'].shift(1))
    low_close = abs(feature_df['Low'] - feature_df['Close'].shift(1))
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    feature_df[f'ATR_{period}'] = true_range.rolling(window=period).mean()
    feature_df[f'ATR_Ratio_{period}'] = feature_df[f'ATR_{period}'] / feature_df['Close'] * 100

# Volatility using standard deviation
for period in [5, 10, 20, 50]:
    feature_df[f'Volatility_{period}'] = feature_df['Close'].pct_change().rolling(window=period).std() * np.sqrt(period)
    feature_df[f'Normalized_Vol_{period}'] = feature_df[f'Volatility_{period}'] / feature_df[f'Volatility_{period}'].rolling(window=100).mean()

# Garman-Klass volatility estimator
feature_df['GK_Volatility'] = np.sqrt(
    0.5 * (np.log(feature_df['High'] / feature_df['Low'])) ** 2 -
    (2 * np.log(2) - 1) * (np.log(feature_df['Close'] / feature_df['Open'])) ** 2
)

# Parkinson's volatility
feature_df['Parkinson_Vol'] = np.sqrt((1 / (4 * np.log(2))) * 
                                     ((np.log(feature_df['High'] / feature_df['Low'])) ** 2))

# Chaikin Volatility
for period in [10, 20]:
    feature_df[f'Chaikin_Vol_{period}'] = (
        (feature_df['High'] - feature_df['Low']).rolling(window=period).mean().pct_change(periods=period) * 100
    )

print("Volatility features created")

## 4. Momentum Indicators

# Relative Strength Index (RSI)
for period in [2, 7, 14, 21]:
    delta = feature_df['Close'].diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs = avg_gain / avg_loss
    feature_df[f'RSI_{period}'] = 100 - (100 / (1 + rs))

# Stochastic Oscillator
for period in [14, 21]:
    feature_df[f'Stoch_K_{period}'] = 100 * ((feature_df['Close'] - feature_df['Low'].rolling(window=period).min()) / 
                                     (feature_df['High'].rolling(window=period).max() - feature_df['Low'].rolling(window=period).min()))
    feature_df[f'Stoch_D_{period}'] = feature_df[f'Stoch_K_{period}'].rolling(window=3).mean()

# ROC (Rate of Change)
for period in [5, 10, 20]:
    feature_df[f'ROC_{period}'] = (feature_df['Close'] / feature_df['Close'].shift(period) - 1) * 100

# Williams %R
for period in [14, 20]:
    feature_df[f'Williams_R_{period}'] = -100 * (
        (feature_df['High'].rolling(window=period).max() - feature_df['Close']) / 
        (feature_df['High'].rolling(window=period).max() - feature_df['Low'].rolling(window=period).min())
    )

# Commodity Channel Index (CCI)
for period in [20]:
    tp = (feature_df['High'] + feature_df['Low'] + feature_df['Close']) / 3
    tp_sma = tp.rolling(window=period).mean()
    mad = tp.rolling(window=period).apply(lambda x: np.mean(np.abs(x - np.mean(x))), raw=True)
    feature_df[f'CCI_{period}'] = (tp - tp_sma) / (0.015 * mad)

print("Momentum features created")

## 5. Trend Indicators

# Average Directional Index (ADX)
for period in [14]:
    # True Range
    high_low = feature_df['High'] - feature_df['Low']
    high_close = abs(feature_df['High'] - feature_df['Close'].shift(1))
    low_close = abs(feature_df['Low'] - feature_df['Close'].shift(1))
    true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    atr = true_range.rolling(window=period).mean()
    
    # Plus Directional Movement (+DM)
    up_move = feature_df['High'] - feature_df['High'].shift(1)
    down_move = feature_df['Low'].shift(1) - feature_df['Low']
    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0)
    plus_dm = pd.Series(plus_dm, index=feature_df.index)
    
    # Minus Directional Movement (-DM)
    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0)
    minus_dm = pd.Series(minus_dm, index=feature_df.index)
    
    # Smoothed +DM and -DM
    smooth_plus_dm = plus_dm.rolling(window=period).sum()
    smooth_minus_dm = minus_dm.rolling(window=period).sum()
    
    # Directional Indicators
    plus_di = 100 * smooth_plus_dm / atr
    minus_di = 100 * smooth_minus_dm / atr
    
    # ADX
    dx = 100 * abs(plus_di - minus_di) / (plus_di + minus_di)
    feature_df[f'ADX_{period}'] = dx.rolling(window=period).mean()
    feature_df[f'Plus_DI_{period}'] = plus_di
    feature_df[f'Minus_DI_{period}'] = minus_di
    feature_df[f'DI_Diff_{period}'] = plus_di - minus_di

# Directional Movement Index (DMI)
for period in [14]:
    feature_df[f'DMI_{period}'] = abs(feature_df[f'Plus_DI_{period}'] - feature_df[f'Minus_DI_{period}']) / (feature_df[f'Plus_DI_{period}'] + feature_df[f'Minus_DI_{period}']) * 100

print("Trend features created")

## 6. Volume Indicators
# On Balance Volume (OBV)
feature_df['OBV_Change'] = np.where(feature_df['Close'] > feature_df['Close'].shift(1), feature_df['Volume'],
                        np.where(feature_df['Close'] < feature_df['Close'].shift(1), -feature_df['Volume'], 0))
feature_df['OBV'] = feature_df['OBV_Change'].cumsum()

# Chaikin Money Flow
for period in [20]:
    mf_multiplier = ((feature_df['Close'] - feature_df['Low']) - (feature_df['High'] - feature_df['Close'])) / (feature_df['High'] - feature_df['Low'])
    mf_volume = mf_multiplier * feature_df['Volume']
    feature_df[f'CMF_{period}'] = mf_volume.rolling(window=period).sum() / feature_df['Volume'].rolling(window=period).sum()

# Volume Oscillator
for short_period, long_period in [(5, 10), (12, 26)]:
    feature_df[f'Volume_Osc_{short_period}_{long_period}'] = (
        feature_df['Volume'].rolling(window=short_period).mean() - 
        feature_df['Volume'].rolling(window=long_period).mean()
    ) / feature_df['Volume'].rolling(window=long_period).mean() * 100

# Volume Rate of Change
for period in [10, 20]:
    feature_df[f'Volume_ROC_{period}'] = (feature_df['Volume'] / feature_df['Volume'].shift(period) - 1) * 100

# Parabolic SAR
def psar(df, iaf=0.02, maxaf=0.2):
    high = df['High']
    low = df['Low']
    close = df['Close']
    
    psar = close.copy()
    bull = True
    af = iaf
    ep = low[0]
    hp = high[0]
    lp = low[0]
    
    for i in range(2, len(df)):
        if bull:
            psar[i] = psar[i-1] + af * (hp - psar[i-1])
        else:
            psar[i] = psar[i-1] + af * (lp - psar[i-1])
        
        reverse = False
        
        if bull:
            if low[i] < psar[i]:
                bull = False
                reverse = True
                psar[i] = hp
                lp = low[i]
                af = iaf
        else:
            if high[i] > psar[i]:
                bull = True
                reverse = True
                psar[i] = lp
                hp = high[i]
                af = iaf
        
        if not reverse:
            if bull:
                if high[i] > hp:
                    hp = high[i]
                    af = min(af + iaf, maxaf)
                if low[i-1] < psar[i]:
                    psar[i] = low[i-1]
                if low[i-2] < psar[i]:
                    psar[i] = low[i-2]
            else:
                if low[i] < lp:
                    lp = low[i]
                    af = min(af + iaf, maxaf)
                if high[i-1] > psar[i]:
                    psar[i] = high[i-1]
                if high[i-2] > psar[i]:
                    psar[i] = high[i-2]
    
    return psar

feature_df['PSAR'] = psar(feature_df)
feature_df['PSAR_Dist'] = (feature_df['Close'] - feature_df['PSAR']) / feature_df['Close'] * 100
feature_df['PSAR_Bull'] = (feature_df['PSAR'] < feature_df['Close']).astype(int)
print("Volume features created")

## 7. Statistical Features and Pattern Recognition

# Skewness and Kurtosis
for period in [20, 50]:
    feature_df[f'Returns_Skewness_{period}'] = feature_df['Percentage_Change'].rolling(window=period).skew()
    feature_df[f'Returns_Kurtosis_{period}'] = feature_df['Percentage_Change'].rolling(window=period).kurt()

# Z-Score
for period in [20, 50]:
    feature_df[f'Price_Z_Score_{period}'] = (feature_df['Close'] - feature_df['Close'].rolling(window=period).mean()) / feature_df['Close'].rolling(window=period).std()
    feature_df[f'Returns_Z_Score_{period}'] = (feature_df['Percentage_Change'] - feature_df['Percentage_Change'].rolling(window=period).mean()) / feature_df['Percentage_Change'].rolling(window=period).std()

# Autocorrelation
for period in [5, 10]:
    feature_df[f'Autocorr_{period}'] = feature_df['Close'].rolling(window=period*2).apply(
        lambda x: pd.Series(x).autocorr(lag=period) if len(x) > period else np.nan
    )

print("Statistical features created")


## 3.1 - Target Feature Engineering

This section creates the target variable for the machine learning models. The target is a 3-class classification variable based on dynamic volatility thresholds:

- **Down (0)**: Price movement below the lower threshold.
- **Up (1)**: Price movement above the upper threshold.
- **Sideways (2)**: Price movement within the thresholds.

The thresholds are calculated using rolling volatility over a defined lookback period, and the future price movement is determined over a prediction horizon. This ensures the target variable reflects market trends while avoiding lookahead bias.

In [ ]:
## Create Target Variable (3-Class Classification with Dynamic Thresholds)

# Set constants for target generation (matching Chosen_Features.ipynb)
LOOKBACK = 20  # Lookback period for volatility calculation
PREDICTION_HORIZON = 5  # Predict price movement 5 periods ahead
MULTIPLIER = 1  # Volatility multiplier for threshold calculation

print(f"Creating target variable with parameters:")
print(f"  - Lookback period: {LOOKBACK}")
print(f"  - Prediction horizon: {PREDICTION_HORIZON}")
print(f"  - Volatility multiplier: {MULTIPLIER}")

# Calculate rolling volatility from the Percentage_Change column
vol = feature_df['Percentage_Change'].rolling(window=LOOKBACK).std()

# Calculate future values (percentage change over prediction horizon)
future_values = feature_df['Close'].pct_change(PREDICTION_HORIZON).shift(-PREDICTION_HORIZON)

# Create dynamic thresholds based on recent volatility
upper_threshold = vol * MULTIPLIER
lower_threshold = -vol * MULTIPLIER

# Generate target variable (3-class classification)
feature_df['Target'] = 2  # Default: sideways (2)
feature_df.loc[future_values > upper_threshold, 'Target'] = 1  # Up (1)
feature_df.loc[future_values < lower_threshold, 'Target'] = 0  # Down (0)

# Remove future data points to avoid lookahead bias
feature_df = feature_df[:-PREDICTION_HORIZON]

# Display target distribution
target_counts = feature_df['Target'].value_counts()
print("\nTarget Distribution:")
print(target_counts)
print(f"Down (0): {target_counts[0]/len(feature_df)*100:.2f}%")
print(f"Up (1): {target_counts[1]/len(feature_df)*100:.2f}%")
print(f"Sideways (2): {target_counts[2]/len(feature_df)*100:.2f}%")

# Use the engineered features as our main data
data = feature_df.copy()

# Remove any NaN values created by indicators
initial_rows = len(data)
data = data.dropna()
dropped_rows = initial_rows - len(data)

print(f"\nDropped {dropped_rows} rows due to NaN in technical indicators")
print(f"Final data shape: {data.shape}")

# Save engineered features to working_data folder
engineered_features_path = 'working_data/engineered_features.csv'
data.to_csv(engineered_features_path)
print(f"Saved engineered features to: {engineered_features_path}")

# Save feature names for later reference
feature_names = [col for col in data.columns if col != 'Target']
with open(os.path.join(run_directory, 'feature_names.pkl'), 'wb') as f:
    pickle.dump(feature_names, f)

print(f"\nTotal features available: {len(feature_names)}")
print(f"Features created: Basic Price, Moving Averages, Volatility, Momentum, Trend, Volume, Statistical")
print(f"Target variable: 'Target' (3-class: 0=Down, 1=Up, 2=Sideways)")

## 4. Data Preprocessing

Perform time-series aware data splitting and feature scaling to prepare for model training.

In [ ]:
# Prepare features and target
X = data[feature_names].copy()
y = data['Target'].copy()

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Time-series aware data splitting (80% train, 10% validation, 10% test)
n_samples = len(data)
train_size = int(0.8 * n_samples)
val_size = int(0.1 * n_samples)

# Split data maintaining temporal order
X_train = X.iloc[:train_size].copy()
X_val = X.iloc[train_size:train_size + val_size].copy()
X_test = X.iloc[train_size + val_size:].copy()

y_train = y.iloc[:train_size].copy()
y_val = y.iloc[train_size:train_size + val_size].copy()
y_test = y.iloc[train_size + val_size:].copy()

print(f"\nTime-series splits:")
print(f"Training: {X_train.shape} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation: {X_val.shape} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test: {X_test.shape} ({len(X_test)/len(X)*100:.1f}%)")

# Check class distribution in each split
print(f"\nClass distribution:")
print(f"Train: {y_train.value_counts().to_dict()}")
print(f"Validation: {y_val.value_counts().to_dict()}")
print(f"Test: {y_test.value_counts().to_dict()}")

In [ ]:
# Feature scaling
print("Applying feature scaling...")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrames for easier handling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_names, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=feature_names, index=X_val.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_names, index=X_test.index)

print("Feature scaling completed!")

# Save preprocessed data
with open(os.path.join(run_directory, 'X_train_scaled.pkl'), 'wb') as f:
    pickle.dump(X_train_scaled, f)
with open(os.path.join(run_directory, 'X_val_scaled.pkl'), 'wb') as f:
    pickle.dump(X_val_scaled, f)
with open(os.path.join(run_directory, 'X_test_scaled.pkl'), 'wb') as f:
    pickle.dump(X_test_scaled, f)
with open(os.path.join(run_directory, 'y_train.pkl'), 'wb') as f:
    pickle.dump(y_train, f)
with open(os.path.join(run_directory, 'y_val.pkl'), 'wb') as f:
    pickle.dump(y_val, f)
with open(os.path.join(run_directory, 'y_test.pkl'), 'wb') as f:
    pickle.dump(y_test, f)
with open(os.path.join(run_directory, 'scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

print("Preprocessed data saved!")

## 4. Baseline Model Training and Testing

Train baseline models (Random Forest, XGBoost, ANN) on all features and evaluate their performance.

In [ ]:
# Define function to evaluate model performance
def evaluate_model(model, X_test, y_test, model_name):
    """Evaluate model performance and return metrics"""
    y_pred = model.predict(X_test)
    
    metrics = {
        'model': model_name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='weighted'),
        'recall': recall_score(y_test, y_pred, average='weighted'),
        'f1_score': f1_score(y_test, y_pred, average='weighted')
    }
    
    return metrics, y_pred

# Store baseline results
baseline_results = []
baseline_models = {}

In [ ]:
# Train Random Forest baseline model
print("Training Random Forest baseline model...")

rf_baseline = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_baseline.fit(X_train_scaled, y_train)

# Evaluate on test set
rf_metrics, rf_pred = evaluate_model(rf_baseline, X_test_scaled, y_test, 'Random Forest Baseline')
baseline_results.append(rf_metrics)
baseline_models['rf'] = rf_baseline

print(f"Random Forest Baseline Results:")
for key, value in rf_metrics.items():
    if key != 'model':
        print(f"  {key}: {value:.4f}")

# Save model
with open(os.path.join(run_directory, 'rf_baseline.pkl'), 'wb') as f:
    pickle.dump(rf_baseline, f)

In [ ]:
# Train XGBoost baseline model
print("Training XGBoost baseline model...")

xgb_baseline = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_baseline.fit(X_train_scaled, y_train)

# Evaluate on test set
xgb_metrics, xgb_pred = evaluate_model(xgb_baseline, X_test_scaled, y_test, 'XGBoost Baseline')
baseline_results.append(xgb_metrics)
baseline_models['xgb'] = xgb_baseline

print(f"XGBoost Baseline Results:")
for key, value in xgb_metrics.items():
    if key != 'model':
        print(f"  {key}: {value:.4f}")

# Save model
with open(os.path.join(run_directory, 'xgb_baseline.pkl'), 'wb') as f:
    pickle.dump(xgb_baseline, f)

In [ ]:
# Train ANN baseline model
print("Training ANN baseline model...")

# Build ANN model
ann_baseline = Sequential([
    layers.Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

ann_baseline.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train the model
history = ann_baseline.fit(
    X_train_scaled, y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=50,
    batch_size=32,
    verbose=0,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
    ]
)

# Evaluate on test set
y_pred_prob = ann_baseline.predict(X_test_scaled)
y_pred_ann = (y_pred_prob > 0.5).astype(int).flatten()

ann_metrics = {
    'model': 'ANN Baseline',
    'accuracy': accuracy_score(y_test, y_pred_ann),
    'precision': precision_score(y_test, y_pred_ann, average='weighted'),
    'recall': recall_score(y_test, y_pred_ann, average='weighted'),
    'f1_score': f1_score(y_test, y_pred_ann, average='weighted')
}

baseline_results.append(ann_metrics)
baseline_models['ann'] = ann_baseline

print(f"ANN Baseline Results:")
for key, value in ann_metrics.items():
    if key != 'model':
        print(f"  {key}: {value:.4f}")

# Save model
ann_baseline.save(os.path.join(run_directory, 'ann_baseline.h5'))

In [ ]:
# Display baseline results summary
print("\n" + "="*60)
print("BASELINE MODEL RESULTS SUMMARY")
print("="*60)

baseline_df = pd.DataFrame(baseline_results)
print(baseline_df.round(4))

# Save baseline results
baseline_df.to_csv(os.path.join(run_directory, 'baseline_results.csv'), index=False)
print("\nBaseline results saved!")

## 5. Feature Selection using Genetic Algorithm (GA)

Apply GA-based feature selection to identify the most relevant features for each model type.

In [ ]:
# GA Feature Selection for Random Forest
print("="*60)
print("GA FEATURE SELECTION FOR RANDOM FOREST")
print("="*60)

# Configure enhanced GA parameters for improved performance
ga_params = {
    'pop_size': 50,                 # Population size
    'crossover_rate': 0.8,          # Base crossover rate (modified by dynamic rates)
    'mutation_rate': 0.2,           # Base mutation rate (modified by dynamic rates)
    'max_generations': 30,          # Maximum generations
    'min_features': 15,              # Minimum features to select
    'max_features': 50,             # Maximum features to select
    'cv_folds': 5,                  # Cross-validation folds
    'adaptive_rates': False,        # DISABLED - Use only dynamic rates for simplicity
    'diversity_threshold': 0.1,     # Lowered for more aggressive diversity management
    'calm_before_storm': 8,         # Reduced for faster adaptation (natural disaster trigger)
    'elite_size': 5,                # Increased elite preservation for better convergence
    'debug': True,                  # Enable detailed logging
    'use_dynamic_rates': True       # Enable dynamic rate strategy (ILM/DHC for pop_size=50)
}

print("Enhanced GA Configuration:")
print(f"  • Population size: {ga_params['pop_size']}")
print(f"  • Strategy: Dynamic rates only (ILM/DHC for small population)")
print(f"  • Feature range: {ga_params['min_features']}-{ga_params['max_features']}")
print(f"  • Elite preservation: {ga_params['elite_size']} individuals")
print(f"  • Natural disaster trigger: {ga_params['calm_before_storm']} stagnant generations")
print(f"  • Diversity threshold: {ga_params['diversity_threshold']}")

# Run GA feature selection for Random Forest
start_time = time.time()

ga_rf = GeneticAlgorithm(
    X_train_scaled, y_train, feature_names,
    model_type='random_forest',
    **ga_params
)

selected_features_rf = ga_rf.run(verbose=True)

rf_ga_time = time.time() - start_time
print(f"\nGA Feature Selection for Random Forest completed in {rf_ga_time:.2f} seconds")
print(f"Selected {len(selected_features_rf[2])} features for Random Forest:")
print(selected_features_rf[2][:10], "..." if len(selected_features_rf[2]) > 10 else "")

# Save selected features
with open(os.path.join(run_directory, 'selected_features_rf.pkl'), 'wb') as f:
    pickle.dump(selected_features_rf, f)

# Plot convergence for Random Forest GA
ga_rf.plot_convergence()

In [ ]:
# GA Feature Selection for XGBoost
print("="*60)
print("GA FEATURE SELECTION FOR XGBOOST")
print("="*60)

# Run GA feature selection for XGBoost
start_time = time.time()

ga_xgb = GeneticAlgorithm(
    X_train_scaled, y_train, feature_names,
    model_type='xgboost',
    **ga_params
)

selected_features_xgb = ga_xgb.run(verbose=True)

xgb_ga_time = time.time() - start_time
print(f"\nGA Feature Selection for XGBoost completed in {xgb_ga_time:.2f} seconds")
print(f"Selected {len(selected_features_xgb)} features for XGBoost:")
print(selected_features_xgb[:10], "..." if len(selected_features_xgb) > 10 else "")

# Save selected features
with open(os.path.join(run_directory, 'selected_features_xgb.pkl'), 'wb') as f:
    pickle.dump(selected_features_xgb, f)

In [ ]:
# GA Feature Selection for ANN
print("="*60)
print("GA FEATURE SELECTION FOR ANN")
print("="*60)

# Run GA feature selection for ANN (uses Random Forest for fitness evaluation)
start_time = time.time()

ga_ann = GeneticAlgorithm(
    X_train_scaled, y_train, feature_names,
    model_type='ann',
    **ga_params
)

selected_features_ann = ga_ann.run(verbose=True)

ann_ga_time = time.time() - start_time
print(f"\nGA Feature Selection for ANN completed in {ann_ga_time:.2f} seconds")
print(f"Selected {len(selected_features_ann)} features for ANN:")
print(selected_features_ann[:10], "..." if len(selected_features_ann) > 10 else "")

# Save selected features
with open(os.path.join(run_directory, 'selected_features_ann.pkl'), 'wb') as f:
    pickle.dump(selected_features_ann, f)

In [ ]:
# Analyze feature selection results
print("\n" + "="*60)
print("FEATURE SELECTION ANALYSIS")
print("="*60)

selection_summary = {
    'Random Forest': len(selected_features_rf),
    'XGBoost': len(selected_features_xgb),
    'ANN': len(selected_features_ann)
}

print("Selected feature counts:")
for model, count in selection_summary.items():
    reduction = (1 - count/len(feature_names)) * 100
    print(f"  {model}: {count} features ({reduction:.1f}% reduction)")

# Find common features across models
common_features = set(selected_features_rf[2]) & set(selected_features_xgb[2]) & set(selected_features_ann[2])
print(f"\nCommon features across all models: {len(common_features)}")
if len(common_features) > 0:
    print("Common features:", list(common_features)[:10], "..." if len(common_features) > 10 else "")

## 6. Feature Engineering using Genetic Algorithm (GA)

Use GA to engineer new optimized features (Adaptive Moving Average, Fractal Dimension Indicator) and add them to the selected feature subsets.

In [ ]:
# Prepare data for feature engineering (use original unscaled data)
print("="*60)
print("GA FEATURE ENGINEERING")
print("="*60)

# Use the original data for feature engineering
engineering_data = data[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
engineering_target = data['Target'].copy()

print(f"Data for feature engineering: {engineering_data.shape}")
print(f"Target distribution: {engineering_target.value_counts().to_dict()}")

In [ ]:
# Engineer Adaptive Moving Average (AMA) using GA
print("Engineering Adaptive Moving Average (AMA)...")

start_time = time.time()

# Configure GA parameters for feature engineering
ga_fe_params = {
    'pop_size': 30,
    'max_generations': 20,
    'crossover_rate': 0.8,
    'mutation_rate': 0.2,
    'cv_folds': 3,
    'random_state': 42
}

# Engineer AMA feature
ama_feature, ama_details = engineer_adaptive_moving_average(
    engineering_data, engineering_target, **ga_fe_params
)

ama_time = time.time() - start_time
print(f"AMA engineering completed in {ama_time:.2f} seconds")
print(f"Best AMA parameters: {ama_details['best_parameters']}")
print(f"AMA fitness: {ama_details['best_fitness']:.4f}")

# Add AMA to the dataset
ama_feature_name = 'AMA_GA_Optimized'

In [ ]:
# Engineer Fractal Dimension Indicator (FDI) using GA
print("Engineering Fractal Dimension Indicator (FDI)...")

start_time = time.time()

# Engineer FDI feature
fdi_feature, fdi_details = engineer_fractal_dimension_indicator(
    engineering_data, engineering_target, **ga_fe_params
)

fdi_time = time.time() - start_time
print(f"FDI engineering completed in {fdi_time:.2f} seconds")
print(f"Best FDI parameters: {fdi_details['best_parameters']}")
print(f"FDI fitness: {fdi_details['best_fitness']:.4f}")

# Add FDI to the dataset
fdi_feature_name = 'FDI_GA_Optimized'

In [ ]:
# Combine engineered features with selected features
print("Combining engineered features with selected feature subsets...")

# Ensure engineered features have the same length as original data
if len(ama_feature) != len(data):
    print(f"Adjusting AMA feature length from {len(ama_feature)} to {len(data)}")
    ama_feature = ama_feature[:len(data)]

if len(fdi_feature) != len(data):
    print(f"Adjusting FDI feature length from {len(fdi_feature)} to {len(data)}")
    fdi_feature = fdi_feature[:len(data)]

# Add engineered features to the dataset splits
# First, add to the original data
data_with_engineered = data.copy()
data_with_engineered[ama_feature_name] = ama_feature
data_with_engineered[fdi_feature_name] = fdi_feature

# Remove any NaN values that might have been introduced
data_with_engineered = data_with_engineered.dropna()

print(f"Data with engineered features shape: {data_with_engineered.shape}")

# Save engineered features details
engineered_features_info = {
    'ama_feature_name': ama_feature_name,
    'ama_details': ama_details,
    'fdi_feature_name': fdi_feature_name,
    'fdi_details': fdi_details
}

with open(os.path.join(run_directory, 'engineered_features_info.pkl'), 'wb') as f:
    pickle.dump(engineered_features_info, f)

In [ ]:
# Create enhanced feature subsets for each model
print("Creating enhanced feature subsets...")

# Combine selected features with engineered features for each model
enhanced_features_rf = list(selected_features_rf[2]) + [ama_feature_name, fdi_feature_name]
enhanced_features_xgb = list(selected_features_xgb[2]) + [ama_feature_name, fdi_feature_name]
enhanced_features_ann = list(selected_features_ann[2]) + [ama_feature_name, fdi_feature_name]

print(f"Enhanced RF features: {len(enhanced_features_rf)} ({len(selected_features_rf)} + 2 engineered)")
print(f"Enhanced XGBoost features: {len(enhanced_features_xgb)} ({len(selected_features_xgb)} + 2 engineered)")
print(f"Enhanced ANN features: {len(enhanced_features_ann)} ({len(selected_features_ann)} + 2 engineered)")

# Save enhanced feature lists
enhanced_features = {
    'rf': enhanced_features_rf,
    'xgb': enhanced_features_xgb,
    'ann': enhanced_features_ann
}

with open(os.path.join(run_directory, 'enhanced_features.pkl'), 'wb') as f:
    pickle.dump(enhanced_features, f)

print("Enhanced feature subsets saved!")

## 7. Enhanced Model Training and Testing

Train models using the combined selected and engineered features, then evaluate their performance.

In [ ]:
# Prepare enhanced datasets
print("Preparing enhanced datasets...")

# Re-split the data with engineered features (maintaining temporal order)
n_samples_new = len(data_with_engineered)
train_size_new = int(0.8 * n_samples_new)
val_size_new = int(0.1 * n_samples_new)

# Create enhanced datasets for each model
enhanced_datasets = {}

for model_name, features in enhanced_features.items():
    print(f"Preparing {model_name} enhanced dataset with {len(features)} features...")
    
    # Extract features from data with engineered features
    X_enhanced = data_with_engineered[features].copy()
    y_enhanced = data_with_engineered['Target'].copy()
    
    # Time-series split
    X_train_enh = X_enhanced.iloc[:train_size_new]
    X_val_enh = X_enhanced.iloc[train_size_new:train_size_new + val_size_new]
    X_test_enh = X_enhanced.iloc[train_size_new + val_size_new:]
    
    y_train_enh = y_enhanced.iloc[:train_size_new]
    y_val_enh = y_enhanced.iloc[train_size_new:train_size_new + val_size_new]
    y_test_enh = y_enhanced.iloc[train_size_new + val_size_new:]
    
    # Scale features
    scaler_enh = StandardScaler()
    X_train_enh_scaled = scaler_enh.fit_transform(X_train_enh)
    X_val_enh_scaled = scaler_enh.transform(X_val_enh)
    X_test_enh_scaled = scaler_enh.transform(X_test_enh)
    
    # Convert back to DataFrames
    X_train_enh_scaled = pd.DataFrame(X_train_enh_scaled, columns=features, index=X_train_enh.index)
    X_val_enh_scaled = pd.DataFrame(X_val_enh_scaled, columns=features, index=X_val_enh.index)
    X_test_enh_scaled = pd.DataFrame(X_test_enh_scaled, columns=features, index=X_test_enh.index)
    
    enhanced_datasets[model_name] = {
        'X_train': X_train_enh_scaled,
        'X_val': X_val_enh_scaled,
        'X_test': X_test_enh_scaled,
        'y_train': y_train_enh,
        'y_val': y_val_enh,
        'y_test': y_test_enh,
        'scaler': scaler_enh
    }

print("Enhanced datasets prepared!")

In [ ]:
# Train enhanced Random Forest model
print("="*60)
print("TRAINING ENHANCED RANDOM FOREST MODEL")
print("="*60)

rf_data = enhanced_datasets['rf']

rf_enhanced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_enhanced.fit(rf_data['X_train'], rf_data['y_train'])

# Evaluate enhanced Random Forest
rf_enh_metrics, rf_enh_pred = evaluate_model(
    rf_enhanced, rf_data['X_test'], rf_data['y_test'], 'Random Forest Enhanced'
)

print(f"Enhanced Random Forest Results:")
for key, value in rf_enh_metrics.items():
    if key != 'model':
        print(f"  {key}: {value:.4f}")

# Save enhanced model
with open(os.path.join(run_directory, 'rf_enhanced.pkl'), 'wb') as f:
    pickle.dump(rf_enhanced, f)
with open(os.path.join(run_directory, 'scaler_rf_enhanced.pkl'), 'wb') as f:
    pickle.dump(rf_data['scaler'], f)

In [ ]:
# Train enhanced XGBoost model
print("="*60)
print("TRAINING ENHANCED XGBOOST MODEL")
print("="*60)

xgb_data = enhanced_datasets['xgb']

xgb_enhanced = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

xgb_enhanced.fit(xgb_data['X_train'], xgb_data['y_train'])

# Evaluate enhanced XGBoost
xgb_enh_metrics, xgb_enh_pred = evaluate_model(
    xgb_enhanced, xgb_data['X_test'], xgb_data['y_test'], 'XGBoost Enhanced'
)

print(f"Enhanced XGBoost Results:")
for key, value in xgb_enh_metrics.items():
    if key != 'model':
        print(f"  {key}: {value:.4f}")

# Save enhanced model
with open(os.path.join(run_directory, 'xgb_enhanced.pkl'), 'wb') as f:
    pickle.dump(xgb_enhanced, f)
with open(os.path.join(run_directory, 'scaler_xgb_enhanced.pkl'), 'wb') as f:
    pickle.dump(xgb_data['scaler'], f)

In [ ]:
# Train enhanced ANN model
print("="*60)
print("TRAINING ENHANCED ANN MODEL")
print("="*60)

ann_data = enhanced_datasets['ann']

# Build enhanced ANN model
ann_enhanced = Sequential([
    layers.Dense(128, activation='relu', input_shape=(ann_data['X_train'].shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

ann_enhanced.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Train enhanced ANN
history_enhanced = ann_enhanced.fit(
    ann_data['X_train'], ann_data['y_train'],
    validation_data=(ann_data['X_val'], ann_data['y_val']),
    epochs=50,
    batch_size=32,
    verbose=0,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
    ]
)

# Evaluate enhanced ANN
y_pred_enh_prob = ann_enhanced.predict(ann_data['X_test'])
y_pred_enh_ann = (y_pred_enh_prob > 0.5).astype(int).flatten()

ann_enh_metrics = {
    'model': 'ANN Enhanced',
    'accuracy': accuracy_score(ann_data['y_test'], y_pred_enh_ann),
    'precision': precision_score(ann_data['y_test'], y_pred_enh_ann, average='weighted'),
    'recall': recall_score(ann_data['y_test'], y_pred_enh_ann, average='weighted'),
    'f1_score': f1_score(ann_data['y_test'], y_pred_enh_ann, average='weighted')
}

print(f"Enhanced ANN Results:")
for key, value in ann_enh_metrics.items():
    if key != 'model':
        print(f"  {key}: {value:.4f}")

# Save enhanced model
ann_enhanced.save(os.path.join(run_directory, 'ann_enhanced.h5'))
with open(os.path.join(run_directory, 'scaler_ann_enhanced.pkl'), 'wb') as f:
    pickle.dump(ann_data['scaler'], f)

## 8. Comparison of Baseline and Enhanced Models

Compare the performance of baseline models vs enhanced models with GA-selected and engineered features.

In [ ]:
# Compile all results for comparison
print("="*60)
print("PERFORMANCE COMPARISON: BASELINE VS ENHANCED")
print("="*60)

# Collect enhanced results
enhanced_results = [rf_enh_metrics, xgb_enh_metrics, ann_enh_metrics]

# Create comprehensive comparison
all_results = baseline_results + enhanced_results
comparison_df = pd.DataFrame(all_results)

print("Complete Results Comparison:")
print(comparison_df.round(4))

# Calculate improvements
improvements = []
model_pairs = [
    ('Random Forest Baseline', 'Random Forest Enhanced'),
    ('XGBoost Baseline', 'XGBoost Enhanced'),
    ('ANN Baseline', 'ANN Enhanced')
]

for baseline_name, enhanced_name in model_pairs:
    baseline_row = comparison_df[comparison_df['model'] == baseline_name].iloc[0]
    enhanced_row = comparison_df[comparison_df['model'] == enhanced_name].iloc[0]
    
    improvement = {
        'model_pair': f"{baseline_name} → {enhanced_name}",
        'accuracy_improvement': enhanced_row['accuracy'] - baseline_row['accuracy'],
        'precision_improvement': enhanced_row['precision'] - baseline_row['precision'],
        'recall_improvement': enhanced_row['recall'] - baseline_row['recall'],
        'f1_improvement': enhanced_row['f1_score'] - baseline_row['f1_score']
    }
    improvements.append(improvement)

improvements_df = pd.DataFrame(improvements)
print("\n" + "="*60)
print("IMPROVEMENTS (Enhanced - Baseline)")
print("="*60)
print(improvements_df.round(4))

In [ ]:
# Visualize performance comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Baseline vs Enhanced Models Comparison', fontsize=16, fontweight='bold')

metrics = ['accuracy', 'precision', 'recall', 'f1_score']
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']

for idx, (metric, metric_name) in enumerate(zip(metrics, metric_names)):
    ax = axes[idx // 2, idx % 2]
    
    # Prepare data for plotting
    models = ['Random Forest', 'XGBoost', 'ANN']
    baseline_values = [comparison_df[comparison_df['model'].str.contains(model) & 
                                   comparison_df['model'].str.contains('Baseline')][metric].values[0] 
                      for model in models]
    enhanced_values = [comparison_df[comparison_df['model'].str.contains(model) & 
                                   comparison_df['model'].str.contains('Enhanced')][metric].values[0] 
                      for model in models]
    
    # Create bar plot
    x = np.arange(len(models))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, baseline_values, width, label='Baseline', alpha=0.8)
    bars2 = ax.bar(x + width/2, enhanced_values, width, label='Enhanced', alpha=0.8)
    
    ax.set_xlabel('Models')
    ax.set_ylabel(metric_name)
    ax.set_title(f'{metric_name} Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.001,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(run_directory, 'performance_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance analysis for enhanced models
print("="*60)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*60)

# Random Forest feature importance
rf_importance = pd.DataFrame({
    'feature': enhanced_features['rf'],
    'importance': rf_enhanced.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Random Forest Features:")
print(rf_importance.head(10))

# XGBoost feature importance
xgb_importance = pd.DataFrame({
    'feature': enhanced_features['xgb'],
    'importance': xgb_enhanced.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 XGBoost Features:")
print(xgb_importance.head(10))

# Save feature importance
rf_importance.to_csv(os.path.join(run_directory, 'rf_feature_importance.csv'), index=False)
xgb_importance.to_csv(os.path.join(run_directory, 'xgb_feature_importance.csv'), index=False)

# Check if engineered features are in top features
ama_rank_rf = rf_importance[rf_importance['feature'] == ama_feature_name].index[0] + 1 if ama_feature_name in rf_importance['feature'].values else 'Not found'
fdi_rank_rf = rf_importance[rf_importance['feature'] == fdi_feature_name].index[0] + 1 if fdi_feature_name in rf_importance['feature'].values else 'Not found'

ama_rank_xgb = xgb_importance[xgb_importance['feature'] == ama_feature_name].index[0] + 1 if ama_feature_name in xgb_importance['feature'].values else 'Not found'
fdi_rank_xgb = xgb_importance[xgb_importance['feature'] == fdi_feature_name].index[0] + 1 if fdi_feature_name in xgb_importance['feature'].values else 'Not found'

print(f"\nEngineered Features Ranking:")
print(f"AMA (Random Forest): {ama_rank_rf}")
print(f"FDI (Random Forest): {fdi_rank_rf}")
print(f"AMA (XGBoost): {ama_rank_xgb}")
print(f"FDI (XGBoost): {fdi_rank_xgb}")

## 9. Save Results and Metadata

Save all models, results, selected features, and metadata to the timestamped run directory.

In [ ]:
# Save comprehensive results
print("="*60)
print("SAVING COMPREHENSIVE RESULTS")
print("="*60)

# Save comparison results
comparison_df.to_csv(os.path.join(run_directory, 'performance_comparison.csv'), index=False)
improvements_df.to_csv(os.path.join(run_directory, 'improvements_summary.csv'), index=False)

# Create comprehensive summary
summary = {
    'run_timestamp': timestamp,
    'run_directory': run_directory,
    'total_original_features': len(feature_names),
    'baseline_results': baseline_results,
    'enhanced_results': enhanced_results,
    'selected_features_count': {
        'rf': len(selected_features_rf),
        'xgb': len(selected_features_xgb),
        'ann': len(selected_features_ann)
    },
    'enhanced_features_count': {
        'rf': len(enhanced_features_rf),
        'xgb': len(enhanced_features_xgb),
        'ann': len(enhanced_features_ann)
    },
    'ga_parameters': ga_params,
    'feature_engineering_parameters': ga_fe_params,
    'engineered_features_info': engineered_features_info,
    'execution_times': {
        'rf_ga_selection': rf_ga_time,
        'xgb_ga_selection': xgb_ga_time,
        'ann_ga_selection': ann_ga_time,
        'ama_engineering': ama_time,
        'fdi_engineering': fdi_time
    }
}

# Save comprehensive summary
with open(os.path.join(run_directory, 'run_summary.pkl'), 'wb') as f:
    pickle.dump(summary, f)

# Save as JSON for human readability (excluding non-serializable objects)
json_summary = {
    'run_timestamp': timestamp,
    'run_directory': run_directory,
    'total_original_features': len(feature_names),
    'selected_features_count': summary['selected_features_count'],
    'enhanced_features_count': summary['enhanced_features_count'],
    'ga_parameters': ga_params,
    'feature_engineering_parameters': ga_fe_params,
    'execution_times': summary['execution_times'],
    'baseline_results': baseline_results,
    'enhanced_results': enhanced_results
}

with open(os.path.join(run_directory, 'run_summary.json'), 'w') as f:
    json.dump(json_summary, f, indent=2)

print(f"Comprehensive results saved to: {run_directory}")

In [ ]:
# Create final summary report
print("="*80)
print("FINAL SUMMARY REPORT")
print("="*80)

print(f"Run completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Results saved in: {run_directory}")
print(f"Total execution time: {time.time() - time.time():.2f} seconds")

print(f"\nDataset Summary:")
print(f"  • Original features: {len(feature_names)}")
print(f"  • Training samples: {len(X_train_scaled)}")
print(f"  • Validation samples: {len(X_val_scaled)}")
print(f"  • Test samples: {len(X_test_scaled)}")

print(f"\nFeature Selection Results:")
for model, count in summary['selected_features_count'].items():
    reduction = (1 - count/len(feature_names)) * 100
    print(f"  • {model.upper()}: {count} features ({reduction:.1f}% reduction)")

print(f"\nFeature Engineering Results:")
print(f"  • AMA optimized parameters: {ama_details['best_parameters']}")
print(f"  • FDI optimized parameters: {fdi_details['best_parameters']}")

print(f"\nPerformance Improvements:")
for _, row in improvements_df.iterrows():
    model_name = row['model_pair'].split(' → ')[0].replace(' Baseline', '')
    print(f"  • {model_name}:")
    # Ensure the row contains the expected keys before accessing
    if 'accuracy_improvement' in row and 'f1_improvement' in row:
        print(f"    - Accuracy: {row['accuracy_improvement']:+.4f}")
        print(f"    - F1-Score: {row['f1_improvement']:+.4f}")
    else:
        print(f"    - Accuracy: Key not found")
        print(f"    - F1-Score: Key not found")
        
print(f"\nFiles Saved:")
saved_files = [f for f in os.listdir(run_directory) if not f.startswith('.')]
for file in sorted(saved_files):
    print(f"  • {file}")

print("\n" + "="*80)
print("WORKFLOW COMPLETED SUCCESSFULLY!")
print("="*80)